# Kafka Producers

![Alt Text](/home/student/A2/FIT3182_A2/A2/34900403_33524815_assignment02/visuals/kafka_workflow_diagram.png)

First, import the the kafka3, pandas and time libraries. The KafkaProducer class and the sleep function would come from the kafak3 and time libraries respectively. From pymongo specifically you would need to import the MongoClient class. Import the json module as well and configure the host IP.

In [49]:
# Import necessary modules, classes and functions
from kafka3 import KafkaProducer
import pandas as pd
from time import sleep
import json

# Configure Host IP
hostip = '192.168.1.108'

The below function establishes a connection to the Kafka broker. This instantiates a KafkaProducer using the configured host IP address and port (9092).

In [50]:
def connect_kafka_producer():
    _producer = None
    try:
        _producer = KafkaProducer(bootstrap_servers=[f'{hostip}:9092'],
                                  api_version=(0, 10))
    except Exception as ex:
        print('Exception while connecting Kafka.')
        print(str(ex))
    
    return _producer

The below function establishes a connection to the Kafka broker. The message key and value payloads are casted to raw bytes using UTF-8 encoding, then pushed to the specified topic and finally, the function clears the network buffers.

In [51]:
def publish_message(producer_instance, topic_name, key, value):
    try:
        key_bytes = bytes(key, encoding='utf-8')
        value_bytes = bytes(value, encoding='utf-8')
        producer_instance.send(topic_name, key=key_bytes, value=value_bytes)
        producer_instance.flush()
        print('Message published successfully. Data: ' + str(value))
    except Exception as ex:
        print('Exception in publishing message.')
        print(str(ex))

Determine names for each of the three topics, one for each camera.

In [52]:
topic_a = 'camera_event_a'
topic_b = 'camera_event_b'
topic_c = 'camera_event_c'

Establish three independent producers, again, one for each camera.

In [53]:
producer_a = connect_kafka_producer()
producer_b = connect_kafka_producer()
producer_c = connect_kafka_producer()

Convert the CSV files containing the data for camera events for three different cameras into pandas dataframes.

In [54]:
# Obtain path directory locations for each CSV file
camera_event_a_path = "/home/student/A2/FIT3182_A2/A2/34900403_33524815_assignment02/data/camera_event_A.csv"
camera_event_b_path = "/home/student/A2/FIT3182_A2/A2/34900403_33524815_assignment02/data/camera_event_B.csv"
camera_event_c_path = "/home/student/A2/FIT3182_A2/A2/34900403_33524815_assignment02/data/camera_event_C.csv"

# Convert the CSV files' content into pandas dataframes
camera_data_a = pd.read_csv(camera_event_a_path)
camera_data_b = pd.read_csv(camera_event_b_path)
camera_data_c = pd.read_csv(camera_event_c_path)

Initialize pointer trackers, localized storage arrays and state flag variables to synchronize/desync multi-stream batches across the three camera datasets.

In [55]:
# Track the current target batch ID sequence
curr_batch_id = 1

# Pointers start at 0
pointer_a = 0
pointer_b = 0
pointer_c = 0

# Arrays to compile row dictionaries for the active batch
batch_data_a = []
batch_data_b = []
batch_data_c = []

# Readiness flags start at False
batch_a_ok= False
batch_b_ok = False
batch_c_ok = False

The streaming loop scans through all the datasets in chronological order in a synchronized pointer-based manner. Rows are aggregate based on the active batch ID, stops the processing in the event of a boundary shift and converts the collected records into JSON before broadcasting them to their respective Kafka topics. Batches are also broadcast 1 second apart.

### Note
Only run this after running the last-most code cell from streaming_app.ipynb if you want the published messages to be added to the MongoDB collection.

In [56]:
def connect_kafka_producer():
    _producer = None
    try:
        _producer = KafkaProducer(bootstrap_servers=[f'{hostip}:9092'],
                                  api_version=(0, 10))
    except Exception as ex:
        print('Exception while connecting Kafka.')
        print(str(ex))
    
    return _producer

def publish_message(producer_instance, topic_name, key, value):
    try:
        key_bytes = bytes(key, encoding='utf-8')
        value_bytes = bytes(value, encoding='utf-8')
        producer_instance.send(topic_name, key=key_bytes, value=value_bytes)
        producer_instance.flush()
        print('Message published successfully. Data: ' + str(value))
    except Exception as ex:
        print('Exception in publishing message.')
        print(str(ex))



topic_a = 'camera_event_a'
topic_b = 'camera_event_b'
topic_c = 'camera_event_c'

producer_a = connect_kafka_producer()
producer_b = connect_kafka_producer()
producer_c = connect_kafka_producer()

camera_event_a_path = "/home/student/A2/FIT3182_A2/A2/34900403_33524815_assignment02/data/camera_event_A.csv"
camera_event_b_path = "/home/student/A2/FIT3182_A2/A2/34900403_33524815_assignment02/data/camera_event_B.csv"
camera_event_c_path = "/home/student/A2/FIT3182_A2/A2/34900403_33524815_assignment02/data/camera_event_C.csv"

camera_data_a = pd.read_csv(camera_event_a_path)
camera_data_b = pd.read_csv(camera_event_b_path)
camera_data_c = pd.read_csv(camera_event_c_path)

curr_batch_id = 1

pointer_a = 0
pointer_b = 0
pointer_c = 0

batch_data_a = []
batch_data_b = []
batch_data_c = []

batch_a_ok= False
batch_b_ok = False
batch_c_ok = False

while pointer_a < len(camera_data_a) or pointer_b < len(camera_data_b) or pointer_c < len(camera_data_c):

    # Gather batches of same batch ID across stream A
    if pointer_a < len(camera_data_a):
        # If a batch with different ID is found, set the batch_id to the new batch's ID
        if camera_data_a.iloc[pointer_a]["batch_id"] != curr_batch_id:
            batch_a_ok = True
        # If there are still records remaining in the current batch
        if not batch_a_ok:
            data = camera_data_a.iloc[pointer_a].to_dict()
            batch_data_a.append(data)
            pointer_a += 1
    
    # Gather batches of same batch ID across stream B
    if pointer_b < len(camera_data_b):
        if camera_data_b.iloc[pointer_b]["batch_id"] != curr_batch_id:
            batch_b_ok = True
        
        if not batch_b_ok:
            data = camera_data_b.iloc[pointer_b].to_dict()
            batch_data_b.append(data)
            pointer_b += 1

    # Gather batches of same batch ID across stream C
    if pointer_c < len(camera_data_c):
        if camera_data_c.iloc[pointer_c]["batch_id"] != curr_batch_id:
            batch_c_ok = True

        if not batch_c_ok:
            data = camera_data_c.iloc[pointer_c].to_dict()
            batch_data_c.append(data)
            pointer_c += 1
        
    # Once boundary flags reveal that all streams have finished gathering the current batch ID
    if batch_a_ok and batch_b_ok and batch_c_ok:

        # Convert the lists of records into JSON blocks
        batch_data_a = json.dumps(batch_data_a)
        batch_data_b = json.dumps(batch_data_b)
        batch_data_c = json.dumps(batch_data_c)

        # Obtain the Kafka key
        kafka_key = str(curr_batch_id)
        
        # Stream the compiled payloads out to their corresponding Kafka broker endpoints
        publish_message(producer_a, topic_a, kafka_key, batch_data_a)
        publish_message(producer_b, topic_b, kafka_key, batch_data_b)
        publish_message(producer_c, topic_c, kafka_key, batch_data_c)
        
        # Reset the batch arrays and readiness flags
        batch_data_a = []
        batch_data_b = []
        batch_data_c = []
        batch_a_ok= False
        batch_b_ok = False
        batch_c_ok = False

        # Move to the next batch index and pause for 1 second
        curr_batch_id += 1
        sleep(1)


Message published successfully. Data: [{"event_id": "d40c586c-5be6-4743-a1e3-2269d9edaa72", "batch_id": 1, "car_plate": "KRN 7", "camera_id": 1, "timestamp": "2024-01-01T08:00:04", "speed_reading": 77.2}, {"event_id": "85c08e3c-a0b5-45d8-a70c-df8f9a6d5829", "batch_id": 1, "car_plate": "ICE 8", "camera_id": 1, "timestamp": "2024-01-01T08:00:05", "speed_reading": 103.7}, {"event_id": "f5834b79-771b-4931-8da2-a5ad7f4ccd02", "batch_id": 1, "car_plate": "QE 1820", "camera_id": 1, "timestamp": "2024-01-01T08:00:03", "speed_reading": 67.4}, {"event_id": "d0e547bb-c4a7-4750-b7b4-8076e9b47f4f", "batch_id": 1, "car_plate": "CJW 924", "camera_id": 1, "timestamp": "2024-01-01T08:00:01", "speed_reading": 148.3}, {"event_id": "f3162606-1b2e-407f-951d-61d14c0a7b09", "batch_id": 1, "car_plate": "CJP 278", "camera_id": 1, "timestamp": "2024-01-01T08:00:02", "speed_reading": 125.2}, {"event_id": "c69852f7-cd9a-4892-b225-8f8a36ec017b", "batch_id": 1, "car_plate": "ZPG 90", "camera_id": 1, "timestamp": "2

KeyboardInterrupt: 